# Study 2b — Steering Vector Experiments

**Question:** Are the linear representations identified by Study 2a probes *causally* linked to reasoning behaviour? Does adding a category's direction vector to the residual stream during generation change the model's reasoning in the predicted way?

**Method:** Additive steering vectors (probe weight directions, normalised to mean activation magnitude) injected at layer 20 during autoregressive generation. 5 categories × 2 directions × 3 strengths × 40 traces/condition.

**Key result:** [Fill after full run]

In [ ]:
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from scipy.stats import wilcoxon

# Navigate to project root if running from notebooks/
_nb_dir = Path('.').resolve()
if _nb_dir.name == 'notebooks':
    os.chdir(_nb_dir.parent.parent)  # study2b_steering/notebooks/ → project root

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'study2b_steering'
FIGURES_DIR = RESULTS_DIR / 'figures'
PROBE_DIR = PROJECT_ROOT / 'outputs' / 'study2_probes'

MICRO_LABELS = ['ORIENT', 'DESCRIBE', 'SYNTHESIZE', 'HYPO', 'TEST',
                'JUDGE', 'PLAN', 'MONITOR', 'RULE']
STEER_CATEGORIES = ['HYPO', 'TEST', 'JUDGE', 'MONITOR', 'PLAN']

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
print('Results dir:', RESULTS_DIR)
print('Files:', sorted(f.name for f in RESULTS_DIR.glob('*.csv')))

## 1. Experimental Design Overview

In [ ]:
# Load condition summaries
summaries = pd.read_csv(RESULTS_DIR / 'condition_summaries.csv')
print(f'Conditions: {len(summaries)}')
summaries[['condition', 'n_traces', 'completion_rate', 'mean_trace_length',
           'mean_transition_entropy']].round(3)

In [ ]:
# Load steering results (effects vs baseline)
results_path = RESULTS_DIR / 'steering_results.csv'
if results_path.exists():
    effects = pd.read_csv(results_path)
    print(f'Steered conditions: {len(effects)}')
    display(effects[['condition', 'category', 'direction', 'alpha',
                     'n_traces', 'completion_rate', 'mean_n_sentences']].round(3))
else:
    print('No steering_results.csv yet. Run analyse_steering.py first.')
    effects = pd.DataFrame()

## 2. On-Target Steering Effects

Does steering toward category X increase the proportion of X in the generated trace?

In [ ]:
if len(effects) > 0:
    # On-target shift for each condition
    on_target = []
    for _, row in effects.iterrows():
        cat = row['category']
        on_target.append({
            'condition': row['condition'],
            'category': cat,
            'direction': row['direction'],
            'alpha': row['alpha'],
            'shift': row.get(f'shift_{cat}', np.nan),
            'cohens_d': row.get(f'd_{cat}', np.nan),
            'p_value': row.get(f'p_{cat}', np.nan),
        })
    on_target_df = pd.DataFrame(on_target)
    
    # Highlight significant effects
    on_target_df['sig'] = on_target_df['p_value'].apply(
        lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
    
    display(on_target_df.round(4))
else:
    print('No effects data available yet.')

## 3. Off-Target Effects & Orthogonality Validation

The near-orthogonal direction geometry (max cos_sim = 0.01) predicts minimal off-target effects.

In [ ]:
# Load shift matrix
shift_path = RESULTS_DIR / 'category_shift_matrix.csv'
if shift_path.exists():
    shift_df = pd.read_csv(shift_path, index_col=0)
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(shift_df) * 0.6 + 1)))
    vmax = max(abs(shift_df.values.min()), abs(shift_df.values.max()), 0.05)
    
    sns.heatmap(shift_df, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                annot=True, fmt='+.3f', linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Shift from baseline'})
    ax.set_title('Category Distribution Shift (steered - baseline)')
    ax.set_ylabel('Steered Condition')
    ax.set_xlabel('Category')
    plt.tight_layout()
    plt.show()
    
    # Diagonal dominance check
    print('\nDiagonal dominance (on-target > max off-target):')
    for idx in shift_df.index:
        cat = idx.rstrip('+-')
        if cat in shift_df.columns:
            on_target = abs(shift_df.loc[idx, cat])
            off_targets = [abs(shift_df.loc[idx, c]) for c in shift_df.columns if c != cat]
            max_off = max(off_targets) if off_targets else 0
            dominant = on_target > max_off
            print(f'  {idx}: on={on_target:.3f}, max_off={max_off:.3f} -> {"PASS" if dominant else "FAIL"}')
else:
    print('No shift matrix yet.')

## 4. Dose-Response Relationships

Does increasing alpha produce monotonically increasing target category proportion?

In [ ]:
dose_path = RESULTS_DIR / 'dose_response.csv'
if dose_path.exists():
    dose_df = pd.read_csv(dose_path)
    categories = dose_df['category'].unique()
    
    fig, axes = plt.subplots(1, len(categories),
                             figsize=(4 * len(categories), 4), squeeze=False)
    axes = axes[0]
    
    # Get baseline proportions
    baseline_per_trace = None
    for name in ['baseline_per_trace.csv', 'pilot_baseline_per_trace.csv']:
        bp = RESULTS_DIR / name
        if bp.exists():
            baseline_per_trace = pd.read_csv(bp)
            break
    
    for i, cat in enumerate(categories):
        ax = axes[i]
        cat_data = dose_df[dose_df['category'] == cat].sort_values('signed_alpha')
        
        # Baseline at alpha=0
        baseline_val = 0
        if baseline_per_trace is not None:
            col = f'prop_{cat}' if f'prop_{cat}' in baseline_per_trace.columns else cat
            if col in baseline_per_trace.columns:
                baseline_val = baseline_per_trace[col].mean()
        
        alphas = [0] + list(cat_data['signed_alpha'])
        props = [baseline_val] + list(cat_data['target_proportion'])
        
        ax.plot(alphas, props, 'o-', color='steelblue', linewidth=2, markersize=8)
        ax.axhline(y=baseline_val, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle=':', alpha=0.3)
        ax.set_xlabel('Signed alpha')
        ax.set_ylabel(f'{cat} proportion')
        ax.set_title(cat)
    
    fig.suptitle('Dose-Response: Target Category Proportion vs Steering Strength',
                 y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No dose-response data yet.')

## 5. Completion Rate & Trace Length Effects

In [ ]:
if len(effects) > 0 and 'completion_rate' in effects.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Sort by category then direction
    plot_df = effects.sort_values(['category', 'direction', 'alpha'])
    
    # Completion rate
    colors = ['steelblue' if d == 'pos' else 'coral'
              for d in plot_df['direction']]
    x = range(len(plot_df))
    ax1.bar(x, plot_df['completion_rate'], color=colors, alpha=0.8)
    # Add baseline line
    baseline_completion = summaries.loc[
        summaries['condition'].str.contains('baseline'), 'completion_rate'].values
    if len(baseline_completion) > 0:
        ax1.axhline(y=baseline_completion[0], color='black', linestyle='--',
                    linewidth=2, label=f'Baseline ({baseline_completion[0]:.2f})')
    ax1.set_xticks(x)
    ax1.set_xticklabels(plot_df['condition'], rotation=45, ha='right', fontsize=7)
    ax1.set_ylabel('Completion Rate')
    ax1.set_title('Completion Rate by Condition')
    ax1.legend()
    
    # Trace length
    if 'mean_n_sentences' in plot_df.columns:
        ax2.bar(x, plot_df['mean_n_sentences'], color=colors, alpha=0.8)
        baseline_len = summaries.loc[
            summaries['condition'].str.contains('baseline'), 'mean_trace_length'].values
        if len(baseline_len) > 0:
            ax2.axhline(y=baseline_len[0], color='black', linestyle='--',
                        linewidth=2, label=f'Baseline ({baseline_len[0]:.0f})')
        ax2.set_xticks(x)
        ax2.set_xticklabels(plot_df['condition'], rotation=45, ha='right', fontsize=7)
        ax2.set_ylabel('Mean Sentences per Trace')
        ax2.set_title('Trace Length by Condition')
        ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('No effects data with completion rates yet.')

## 6. Accuracy Effects

For completed traces, does steering change answer quality? Requires ground-truth rules.

In [ ]:
# Ground truth rules for tasks 1-4 (from stimuli)
GROUND_TRUTH = {
    1: 'Panels with at least one red cone',
    2: 'All cones point upward',
    3: 'At least one large cone present',
    4: 'Exactly two cones are stacked',
}

# TODO: After full run, evaluate completed traces against ground truth
# Either manually or via LLM judge
print('Accuracy evaluation pending full data collection.')
print('Ground truth rules:', GROUND_TRUTH)

## 7. Venhoff Vector Comparison

Compare our probe-derived steering vectors with Venhoff et al.'s pre-computed difference-of-means vectors.

In [ ]:
# Load our steering directions and Venhoff vectors for comparison
import torch

our_dirs = np.load(str(PROBE_DIR / 'steering_directions_layer20.npz'))

venhoff_path = PROJECT_ROOT / 'steering-thinking-llms' / 'train-steering-vectors' / 'results' / 'vars' / 'mean_vectors_deepseek-r1-distill-llama-8b.pt'
if venhoff_path.exists():
    venhoff = torch.load(str(venhoff_path), map_location='cpu', weights_only=True)
    
    # Compute Venhoff feature vectors at layer 20
    overall_mean = venhoff['overall']['mean'][20].numpy()
    venhoff_cats = ['backtracking', 'uncertainty-estimation', 'example-testing',
                    'adding-knowledge', 'deduction', 'initializing']
    our_map = {'backtracking': 'MONITOR', 'example-testing': 'TEST',
               'deduction': 'JUDGE', 'initializing': 'ORIENT'}
    
    print('Cosine similarity between our directions and Venhoff vectors (layer 20):')
    for vlabel in venhoff_cats:
        v_vec = venhoff[vlabel]['mean'][20].numpy() - overall_mean
        v_vec = v_vec / (np.linalg.norm(v_vec) + 1e-10)
        
        our_label = our_map.get(vlabel)
        if our_label and our_label in our_dirs:
            cos_sim = np.dot(our_dirs[our_label], v_vec)
            print(f'  Venhoff {vlabel:25s} <-> Ours {our_label:10s}: cos_sim = {cos_sim:.4f}')
else:
    print('Venhoff vectors not found at:', venhoff_path)

## 8. Mechanistic Grounding Classification

Combine Study 2a (probe) and Study 2b (steering) evidence to classify each category.

In [ ]:
# Load probe results from Study 2a
probe_f1_path = PROBE_DIR / 'probe_per_category_f1.csv'
if probe_f1_path.exists():
    probe_f1 = pd.read_csv(probe_f1_path)
    print('Probe per-category F1:')
    display(probe_f1)

# Build classification table
if len(effects) > 0 and probe_f1_path.exists():
    classification = []
    
    # Get F1 values (layer 20, mean_pool, C=1.0)
    f1_cols = [c for c in probe_f1.columns if c.endswith('_f1')]
    f1_dict = {}
    for col in f1_cols:
        cat = col.replace('_f1', '')
        # Find the row for layer 20, mean_pool
        mask = (probe_f1['layer'] == 20) & (probe_f1['aggregation'] == 'mean_pool')
        if mask.any():
            f1_dict[cat] = probe_f1.loc[mask, col].values[0]
    
    for cat in MICRO_LABELS:
        probe_score = f1_dict.get(cat, np.nan)
        
        # Get steering effect at alpha=1.0, pos direction
        mask = (effects['category'] == cat) & (effects['direction'] == 'pos') & (effects['alpha'] == 1.0)
        if mask.any():
            shift = effects.loc[mask, f'shift_{cat}'].values[0]
            p = effects.loc[mask, f'p_{cat}'].values[0]
        else:
            shift = np.nan
            p = np.nan
        
        # Classification logic
        if pd.isna(probe_score) or probe_score < 0.5:
            status = 'Performative'
        elif pd.isna(shift):
            status = 'Not tested'
        elif shift > 0.02 and (p < 0.05 or pd.isna(p)):
            status = 'Mechanistically grounded'
        elif shift > 0:
            status = 'Partially grounded'
        else:
            status = 'Performative'
        
        classification.append({
            'Category': cat,
            'Probe F1': round(probe_score, 3) if not pd.isna(probe_score) else '-',
            'Steering Shift': f'{shift:+.3f}' if not pd.isna(shift) else '-',
            'p-value': f'{p:.4f}' if not pd.isna(p) else '-',
            'Classification': status,
        })
    
    class_df = pd.DataFrame(classification)
    display(class_df)
else:
    print('Need both probe and steering results for classification.')
    print('Run the full experiment pipeline first.')

---

*Notebook: `study2_02_steering.ipynb` | Study 2b: Causal Verification via Steering Vectors*